# sessions

> Async client for Jupyter sessions over `/api/sessions`

In [ ]:
#| default_exp sessions

`JupyAsyncSessionsClient` manages standard Jupyter sessions over `/api/sessions`, binding notebook or file paths to kernels on [jupygate](https://github.com/AnswerDotAI/jupygate) or Jupyter Server.

In [ ]:
#| export
from pathlib import Path
from fastcore.basics import patch
from jupyasyncclient.core import KernelApi, JupyAsyncKernelClient

In [ ]:
import shutil, tempfile
from fastcore.test import test_eq
from jupyasyncclient import JupyAsyncFilesClient, start_gateway

## The sessions client

`JupyAsyncSessionsClient` inherits `KernelApi` HTTP plumbing and `jg_spec()`. `create` starts or reuses the session bound to `path`. `patch` updates its path, name, type, or kernel; `rename` is a shorthand for path updates. `delete` removes the session and shuts down its kernel.

In [ ]:
#| export
def _kernel_ref(kernel=None, kernel_name=None, kernel_id=None):
    k = dict(kernel or {})
    if kernel_name is not None: k['name'] = kernel_name
    if kernel_id is not None: k['id'] = kernel_id
    return k or None

class JupyAsyncSessionsClient(KernelApi):
    "Jupyter sessions over `/api/sessions`: bind notebook/file paths to kernels."
    async def list(self):
        "List all active sessions."
        return await self.api.sessions.list_sessions()

    async def get(self, sid):
        "Fetch one session model by `sid`."
        return await self.api.sessions.get_session(sid=sid)

    async def delete(self, sid):
        "Delete session `sid` and shut down its kernel."
        return await self.api.sessions.delete_session(sid=sid)

In [ ]:
#| export
@patch
async def create(
    self:JupyAsyncSessionsClient,
    path, # Notebook or file path relative to the server root
    name=None, # Display name (defaults to the filename)
    type='notebook', # Session type (`'notebook'`, `'file'`, `'console'`, ...)
    kernel_name=None, # Kernelspec name to launch when creating a new kernel
    kernel_id=None, # Existing kernel id to attach to the session
    kernel=None, # Explicit kernel dict (`{'name': ..., 'id': ...}`)
    **kwargs
):
    "Create a session for `path`, or return the existing session for that path."
    p = str(path)
    body = dict(path=p, name=Path(p).name if name is None else name, type=type, **kwargs)
    if (k := _kernel_ref(kernel, kernel_name, kernel_id)) is not None: body['kernel'] = k
    return await self.api.sessions.create_session(body_=body)

@patch
async def patch(
    self:JupyAsyncSessionsClient,
    sid, # Session id
    path=None, # New path
    name=None, # New display name
    type=None, # New session type
    kernel_name=None, # Start and attach a new kernel with this kernelspec name
    kernel_id=None, # Switch session to an existing kernel id
    kernel=None, # Explicit kernel dict
    **kwargs
):
    "Update session `sid` metadata or kernel binding."
    body = dict(kwargs)
    if path is not None: body['path'] = str(path)
    if name is not None: body['name'] = name
    if type is not None: body['type'] = type
    if (k := _kernel_ref(kernel, kernel_name, kernel_id)) is not None: body['kernel'] = k
    return await self.api.sessions.patch_session(sid=sid, body_=body)

@patch
async def rename(self:JupyAsyncSessionsClient, sid, path, name=None):
    "Update session `sid` to track a renamed `path`."
    return await self.patch(sid, path=path, name=name)

`for_path` finds the active session bound to a path. `kernel` opens a ready `JupyAsyncKernelClient` attached to a session (accepting either a session model dict or a session id) without claiming kernel ownership, inheriting authentication and TLS settings from this client:

In [ ]:
#| export
@patch
async def for_path(self:JupyAsyncSessionsClient, path):
    "The active session model bound to `path`, or None."
    p = str(path).strip('/')
    return next((s for s in await self.list() if (s.get('path') or '').strip('/') == p), None)

@patch
async def kernel(self:JupyAsyncSessionsClient, session, timeout=60, **kwargs):
    "Return a ready `JupyAsyncKernelClient` attached to `session` (model dict or session id)."
    s = await self.get(session) if isinstance(session, str) else session
    return await JupyAsyncKernelClient.connect(
        self.base_url, kernel=s['kernel']['id'], token=self.token,
        headers=self._headers, verify=self.verify, timeout=timeout, **kwargs)

Let's start a local `jupygate` process, create a notebook, open a session for it, run code on its kernel, rename the session, and delete it:

In [ ]:
root = Path(tempfile.mkdtemp())
g = start_gateway(('jupygate', '--root', root))
fc, sc = JupyAsyncFilesClient(g.url), JupyAsyncSessionsClient(g.url)
nb = await fc.post('', type='notebook')
s = await sc.create(nb['path'])
test_eq((await sc.create(nb['path']))['id'], s['id'])
s

In [ ]:
async with await sc.kernel(s) as kc:
    assert not kc.owned
    await kc.reply('answer = 42', timeout=15)
moved = await fc.rename(nb['path'], 'moved.ipynb')
s2 = await sc.rename(s['id'], moved['path'], name='moved.ipynb')
test_eq((await sc.for_path('moved.ipynb'))['id'], s['id'])
async with await sc.kernel(s2['id']) as kc:
    test_eq(await kc.eval_expr('answer'), 42)
await sc.delete(s['id'])
test_eq(await sc.for_path('moved.ipynb'), None)
test_eq(await sc.list(), [])

In [ ]:
#| hide
g.stop()
shutil.rmtree(root, ignore_errors=True)

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()